# Exploring chatting with PydanticAI and own FastAPI endpoint

In [2]:
from pydantic_ai import Agent
from constants import MODEL_SMALL  # Nvidia Build model via nvidia_model()

chat_agent = Agent(
    MODEL_SMALL,
    system_prompt="Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.",
)

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-bolt-e4b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

In [3]:
result = await chat_agent.run("hello")
result

                 pydantic-ai v2.62.0 • Python 3.13.7
      / \
     /   \       agent: _ • model: openai:nvidia/nemotron-3.5-bolt-e4b • tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


AgentRunResult(output="Why did the programmer quit his job? 👨\u200d💻 Because he didn't get arrays! 😩😂")

In [4]:
print(result.output)

Why did the programmer quit his job? 👨‍💻 Because he didn't get arrays! 😩😂


In [5]:
result = await chat_agent.run("what did I ask first?")
print(result.output)

Why do programmers prefer dark mode? 🤔 Because light attracts bugs! 🐛💻😂


In [6]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', timestamp=datetime.datetime(2026, 11, 2, 11, 22, 2, 114508, tzinfo=datetime.timezone.utc)), UserPromptPart(content='what did I ask first?', timestamp=datetime.datetime(2026, 11, 2, 11, 22, 2, 114549, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 2, 11, 22, 2, 115814, tzinfo=datetime.timezone.utc), run_id='01a19bfe-affc-73f5-b1e7-85249b4bfb40', conversation_id='01a19bfe-affd-722d-b204-f72ec733077c'),
 ModelResponse(parts=[TextPart(content='Why do programmers prefer dark mode? 🤔 Because light attracts bugs! 🐛💻😂')], usage=RequestUsage(cache_read_tokens=32, details={'reasoning_tokens': 0}, input_tokens=45, output_tokens=18), model_name='nvidia/nemotron-3.5-bolt-e4b', timestamp=datetime.datetime(2026, 11, 2, 11, 22, 2, 564999, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://inte

In [7]:
result = await chat_agent.run("what is the weather in stockholm?")
result

AgentRunResult(output="Why did the programmer quit his job? Because he didn't get arrays! 🤓💻☔️😂")

In [8]:
result2 = await chat_agent.run(
    "what did I ask you first?", message_history=result.all_messages()
)
print(result2.output)

I'd tell ya, but my memory is like a stack without a pop function! 😅💾🤷‍♂️🤣


## Testing out endpoint

Start the API first in a terminal: `uv run uvicorn api:app --reload`

In [9]:
import httpx

result = httpx.post(
    "http://localhost:8000/chat",
    headers={"accept": "application/json", "Content-Type": "application/json"},
    json={"question": "why is the weather gray"},
    timeout=60,
)

result

ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
result.json()["response"]

In [ ]:
result.json().keys()

In [ ]:
msg_history = result.json()["message_history"]
msg_history

In [ ]:
type(msg_history)

In [ ]:
result = httpx.post(
    "http://localhost:8000/chat",
    json={"question": "what did I just ask you?", "message_history": msg_history},
    timeout=60,
)

result.json()["response"]

In [ ]:
result.json()["message_history"][2]